# Fine-tune Qwen2.5-7B-Instruct for PDF Metadata Extraction

This notebook fine-tunes **Qwen2.5-7B-Instruct** on the PDF metadata
extraction dataset using **LoRA** on a Colab Pro A100.

**Resume-safe:** All outputs persist on Google Drive. After a Colab
disconnect, re-import the notebook and click **Runtime > Run all** to
resume from where it left off.

## Pipeline
1. Install dependencies
2. Mount Google Drive & upload data
3. Fine-tune with LoRA (chat-template tokenisation, label masking)
4. Merge LoRA into base model
5. Convert merged model to GGUF Q4_K_M
6. Download final `.gguf` to Drive

## Requirements
- Colab Pro with **A100 GPU** (40-80 GB VRAM)
- `train.jsonl` and `val.jsonl` in Google Drive at `MyDrive/pdf-meta-llm/datasets/`

## Fresh restart
To retrain from scratch, delete `finetuned/`, `merged/`, and `tokenized/`
from `MyDrive/pdf-meta-llm/` in Google Drive, then Run All.

## 1. Install Dependencies

In [ ]:
!pip install -q \
    torch \
    transformers>=4.45.0 \
    peft>=0.12.0 \
    datasets \
    accelerate>=0.33.0 \
    bitsandbytes \
    sentencepiece \
    protobuf

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Mount Google Drive & Load Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ---------- paths (all expensive outputs on Drive) ----------
DRIVE_DIR    = '/content/drive/MyDrive/pdf-meta-llm'
DATA_DIR     = f'{DRIVE_DIR}/datasets'
OUTPUT_DIR   = f'{DRIVE_DIR}/finetuned'   # LoRA checkpoints + final adapter
MERGED_DIR   = f'{DRIVE_DIR}/merged'      # merged HF model
GGUF_DIR     = f'{DRIVE_DIR}/gguf'        # final quantised GGUF

# Local scratch — ephemeral, OK to lose on disconnect
LOCAL_SCRATCH = '/content/scratch'

for d in [OUTPUT_DIR, MERGED_DIR, GGUF_DIR, LOCAL_SCRATCH]:
    os.makedirs(d, exist_ok=True)

In [ ]:
# Verify data files exist
import json
from pathlib import Path

for split in ['train', 'val']:
    path = Path(DATA_DIR) / f'{split}.jsonl'
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. Upload train.jsonl and val.jsonl to "
            f"{DATA_DIR} in Google Drive."
        )
    n_lines = sum(1 for _ in open(path))
    print(f"{split}: {n_lines:,} examples")

# Show a sample
with open(Path(DATA_DIR) / 'train.jsonl') as f:
    sample = json.loads(f.readline())
print(f"\nSample keys: {list(sample.keys())}")
print(f"Messages: {len(sample['messages'])} turns")
print(f"Assistant (ground truth): {sample['messages'][2]['content'][:200]}")

## 3. Fine-tune with LoRA

In [ ]:
import math
import time
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, PeftModel

# Configuration
BASE_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
MAX_SEQ_LEN = 3072      # was 2048 — avoids truncating assistant labels on long PDFs
EPOCHS = 3              # was 2 — dataset is large enough to benefit from more passes
LR = 2e-5              # was 1e-4 — lower LR with more epochs for better convergence
BATCH_SIZE = 4
GRAD_ACCUM = 4          # effective batch = 16

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Vocab size: {tokenizer.vocab_size:,}")

In [ ]:
# Tokenisation with label masking — cached to Drive for resume
from datasets import load_from_disk

TOKENIZED_TRAIN = f'{DRIVE_DIR}/tokenized/train'
TOKENIZED_VAL   = f'{DRIVE_DIR}/tokenized/val'

def tokenize_chat(example):
    messages = example['messages']

    # Full conversation
    full_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    full_ids = tokenizer(
        full_text, truncation=True, max_length=MAX_SEQ_LEN,
        return_attention_mask=True,
    )
    input_ids = full_ids['input_ids']
    attention_mask = full_ids['attention_mask']

    # Prefix (system + user) — loss only on assistant tokens
    prefix_text = tokenizer.apply_chat_template(
        messages[:-1], tokenize=False, add_generation_prompt=True
    )
    prefix_ids = tokenizer(
        prefix_text, truncation=True, max_length=MAX_SEQ_LEN
    )['input_ids']
    prefix_len = len(prefix_ids)

    labels = [-100] * prefix_len + input_ids[prefix_len:]
    labels = (labels + [-100] * len(input_ids))[:len(input_ids)]

    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'labels': labels,
    }

if os.path.isdir(TOKENIZED_TRAIN) and os.path.isdir(TOKENIZED_VAL):
    print('Loading cached tokenized datasets from Drive...')
    train_ds = load_from_disk(TOKENIZED_TRAIN)
    val_ds   = load_from_disk(TOKENIZED_VAL)
    print(f'Loaded: train={len(train_ds):,}  val={len(val_ds):,}')
else:
    train_ds = load_dataset('json', data_files=f'{DATA_DIR}/train.jsonl', split='train')
    val_ds   = load_dataset('json', data_files=f'{DATA_DIR}/val.jsonl', split='train')

    print(f'Tokenising {len(train_ds):,} train + {len(val_ds):,} val examples...')
    train_ds = train_ds.map(tokenize_chat, remove_columns=train_ds.column_names, num_proc=4)
    val_ds   = val_ds.map(tokenize_chat, remove_columns=val_ds.column_names, num_proc=4)

    train_ds = train_ds.filter(lambda x: len(x['input_ids']) >= 50)
    val_ds   = val_ds.filter(lambda x: len(x['input_ids']) >= 50)
    print(f'After filtering: train={len(train_ds):,}  val={len(val_ds):,}')

    # Cache to Drive for resume
    train_ds.save_to_disk(TOKENIZED_TRAIN)
    val_ds.save_to_disk(TOKENIZED_VAL)
    print('Tokenized datasets cached to Drive.')

In [ ]:
# Load model — checkpoint-aware for resume after disconnect
import glob

def find_latest_checkpoint(output_dir):
    """Return path to the latest valid checkpoint-NNNN dir, or None."""
    ckpts = sorted(
        glob.glob(os.path.join(output_dir, 'checkpoint-*')),
        key=lambda p: int(os.path.basename(p).split('-')[-1]),
    )
    for ckpt in reversed(ckpts):
        required = ['adapter_model.safetensors', 'trainer_state.json']
        if all(os.path.exists(os.path.join(ckpt, f)) for f in required):
            return ckpt
        print(f'WARNING: Corrupt checkpoint {ckpt}, skipping.')
    return None

ADAPTER_DONE = os.path.join(OUTPUT_DIR, 'adapter_config.json')
TRAINING_COMPLETE = os.path.exists(ADAPTER_DONE)
RESUME_CKPT = None if TRAINING_COMPLETE else find_latest_checkpoint(OUTPUT_DIR)

if TRAINING_COMPLETE:
    print(f'Training already complete. Final adapter at {OUTPUT_DIR}.')
elif RESUME_CKPT:
    print(f'Found checkpoint: {RESUME_CKPT} — will resume training.')
else:
    print('No checkpoint found — starting fresh.')

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
    attn_implementation='sdpa',
)
model.config.use_cache = False

if RESUME_CKPT:
    # Resume: load LoRA weights from checkpoint
    model = PeftModel.from_pretrained(model, RESUME_CKPT, is_trainable=True)
else:
    # Fresh LoRA — r=32 for more capacity (was r=16)
    lora_config = LoraConfig(
        r=32,
        lora_alpha=64,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                        'gate_proj', 'up_proj', 'down_proj'],
        lora_dropout=0.05,
        bias='none',
        task_type='CAUSAL_LM',
    )
    model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

In [ ]:
# Data collator
from typing import List, Dict, Any

class ChatDataCollator:
    def __init__(self, pad_id):
        self.pad_id = pad_id

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        max_len = max(len(f['input_ids']) for f in features)
        input_ids, attention_mask, labels = [], [], []
        for f in features:
            pad = max_len - len(f['input_ids'])
            input_ids.append(f['input_ids'] + [self.pad_id] * pad)
            attention_mask.append(f['attention_mask'] + [0] * pad)
            labels.append(f['labels'] + [-100] * pad)
        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'labels': torch.tensor(labels, dtype=torch.long),
        }

collator = ChatDataCollator(tokenizer.pad_token_id or tokenizer.eos_token_id)

In [ ]:
# Training — skips if already complete, resumes from checkpoint if available
if TRAINING_COMPLETE:
    print('Training already done. Skipping to next step.')
else:
    effective_batch = BATCH_SIZE * GRAD_ACCUM
    total_steps = math.ceil(len(train_ds) / effective_batch) * EPOCHS
    print(f'Effective batch: {effective_batch}  |  Total steps: ~{total_steps:,}')
    if RESUME_CKPT:
        print(f'Resuming from {RESUME_CKPT}')

    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        lr_scheduler_type='cosine',
        warmup_ratio=0.05,
        weight_decay=0.01,
        logging_steps=10,
        save_strategy='steps',
        save_steps=500,
        save_total_limit=3,
        eval_strategy='steps',
        eval_steps=500,
        bf16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
        report_to='none',
        dataloader_num_workers=2,
        remove_unused_columns=False,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=collator,
    )

    t0 = time.monotonic()
    trainer.train(resume_from_checkpoint=RESUME_CKPT)
    elapsed = time.monotonic() - t0
    print(f'\nTraining completed in {elapsed / 60:.1f} minutes')

    # Save final LoRA adapter
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f'LoRA adapter saved to {OUTPUT_DIR}')

## 4. Merge LoRA into Base Model

In [ ]:
# Merge LoRA — skips if already done
MERGED_MARKER = os.path.join(MERGED_DIR, 'config.json')

if os.path.exists(MERGED_MARKER):
    print(f'Merged model already exists at {MERGED_DIR}. Skipping.')
else:
    # Free GPU memory from training
    if 'model' in dir():
        del model
    if 'trainer' in dir():
        del trainer
    torch.cuda.empty_cache()

    print('Merging LoRA weights...')
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.bfloat16,
        device_map='auto',
        trust_remote_code=True,
    )

    merged = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
    merged = merged.merge_and_unload()

    merged.save_pretrained(MERGED_DIR)
    tokenizer.save_pretrained(MERGED_DIR)
    print(f'Merged model saved to {MERGED_DIR}')

    del base_model, merged
    torch.cuda.empty_cache()

## 5. Convert to GGUF Q4_K_M

In [ ]:
# Build llama.cpp — always local (Drive FUSE strips execute permissions)
import subprocess

LLAMA_QUANTIZE = '/content/llama.cpp/build/bin/llama-quantize'

if os.path.isfile(LLAMA_QUANTIZE):
    print('llama.cpp already built. Skipping.')
else:
    !rm -rf /content/llama.cpp
    !git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp
    !cd /content/llama.cpp && cmake -B build && cmake --build build --config Release -j $(nproc)

In [ ]:
# Install conversion dependencies
!pip install -q gguf numpy sentencepiece

In [ ]:
# Convert HF model to GGUF FP16 — skips if final GGUF already exists
GGUF_NAME = 'qwen2.5-7b-pdfmeta-q4_k_m.gguf'
GGUF_PATH = f'{GGUF_DIR}/{GGUF_NAME}'
FP16_GGUF = f'{LOCAL_SCRATCH}/model-fp16.gguf'

if os.path.exists(GGUF_PATH):
    print(f'Final GGUF already exists at {GGUF_PATH}. Skipping FP16 conversion.')
elif os.path.exists(FP16_GGUF):
    print(f'FP16 GGUF already exists at {FP16_GGUF}. Skipping conversion.')
else:
    !python /content/llama.cpp/convert_hf_to_gguf.py \
        {MERGED_DIR} \
        --outfile {FP16_GGUF} \
        --outtype f16

In [ ]:
# Quantize to Q4_K_M — skips if already done
if os.path.exists(GGUF_PATH):
    size_gb = os.path.getsize(GGUF_PATH) / 1e9
    print(f'Quantized GGUF already exists: {GGUF_PATH} ({size_gb:.2f} GB). Skipping.')
else:
    !{LLAMA_QUANTIZE} \
        {FP16_GGUF} \
        {GGUF_PATH} \
        Q4_K_M

    size_gb = os.path.getsize(GGUF_PATH) / 1e9
    print(f'\nGGUF model: {GGUF_PATH}')
    print(f'Size: {size_gb:.2f} GB')

## 6. Verify the GGUF Model

In [ ]:
# Quick smoke test with llama-cpp-python
!pip install -q llama-cpp-python

In [ ]:
from llama_cpp import Llama, LlamaGrammar

GRAMMAR = r'''
root   ::= "{" ws "\"title\"" ws ":" ws string "," ws "\"authors\"" ws ":" ws array ws "}"
string ::= "\"" ([^"\\] | "\\" .)* "\""
array  ::= "[" ws string (ws "," ws string)* ws "]" | "[" ws "]"
ws     ::= [ \t\n]*
'''

llm = Llama(model_path=GGUF_PATH, n_ctx=4096, verbose=False)
grammar = LlamaGrammar.from_string(GRAMMAR)

test_text = '''On the convergence of stochastic gradient descent
with adaptive learning rates

Jean-Pierre Dupont and Marie Curie-Sklodowska

Abstract. We study the convergence properties of SGD...'''

prompt = (
    '<|im_start|>system\n'
    'You are a metadata extraction assistant for academic mathematics papers.\n'
    'Given the first pages of a PDF, extract the paper\'s title and all author names.\n'
    'Return a JSON object with exactly two keys:\n'
    '  "title" \u2014 the full paper title (string)\n'
    '  "authors" \u2014 all author names (array of strings, each "Firstname Lastname")\n'
    'Rules:\n'
    '- Use the title exactly as it appears in the text (preserve the original language).\n'
    '- Do NOT include affiliations, emails, or other data.\n'
    '- If the text is too short or unclear to determine the title, return {"title": "", "authors": []}.<|im_end|>\n'
    '<|im_start|>user\n'
    f'<PDF text>\n{test_text}\n</PDF text>\n\n'
    'Extract the title and authors from the above academic paper text.<|im_end|>\n'
    '<|im_start|>assistant\n'
)

output = llm.create_completion(
    prompt, max_tokens=256, grammar=grammar, temperature=0.0
)

import json
result = json.loads(output['choices'][0]['text'].strip())
print('Extracted metadata:')
print(json.dumps(result, indent=2, ensure_ascii=False))

del llm  # free memory

## 7. Download Instructions

The GGUF model is saved to Google Drive at:
```
MyDrive/pdf-meta-llm/gguf/qwen2.5-7b-pdfmeta-q4_k_m.gguf
```

To use it locally:
1. Download from Google Drive
2. Place at `~/.mathpdf_models/gguf/qwen2.5-7b-pdfmeta-q4_k_m.gguf`
3. The `LLMMetadataExtractor` will automatically detect and use it

### Resume after disconnect
This notebook is fully resume-safe. If Colab disconnects:
1. Re-open (or re-upload) the notebook
2. Click **Runtime > Run All**
3. It will skip all completed steps and resume from where it left off

### Retrain from scratch
To start a fresh training run, delete these Drive folders before running:
```
MyDrive/pdf-meta-llm/finetuned/
MyDrive/pdf-meta-llm/merged/
MyDrive/pdf-meta-llm/tokenized/
```